**Question 1**

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer
!pip install -q librosa soundfile torchaudio
!pip install -q tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import torch

GCP_BASE = "https://storage.googleapis.com/upload_goai"
DATA_DIR        = Path("data/raw_audio")
PROCESSED_DIR   = Path("data/processed")
OUTPUT_DIR      = Path("models/whisper-small-hindi")
PREDICTIONS_PATH = Path("data/predictions.json")

SAMPLE_RATE     = 16_000
MIN_DURATION    = 0.5
MAX_DURATION    = 30.0
MAX_WORKERS     = 8


MODEL_NAME      = "openai/whisper-small"
LANGUAGE        = "Hindi"
TASK            = "transcribe"

MAX_STEPS             = 4000
TRAIN_BATCH_SIZE      = 16
EVAL_BATCH_SIZE       = 8
EVAL_STEPS            = 500
SAVE_STEPS            = 500
LEARNING_RATE         = 1e-5
WARMUP_STEPS          = 500
GENERATION_MAX_LENGTH = 225    # Whisper default

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create dirs
for d in [DATA_DIR, PROCESSED_DIR, OUTPUT_DIR, Path("data")]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Device   : {DEVICE}")
print(f"Model    : {MODEL_NAME}")
print(f"Data dir : {DATA_DIR}")
print(f"Output   : {OUTPUT_DIR}")

Device   : cuda
Model    : openai/whisper-small
Data dir : data/raw_audio
Output   : models/whisper-small-hindi


In [ ]:
from random import sample
from IPython.core.magics.script import time
import re
import json
import requests
import numpy as np
import soundfile as sf
import librosa
import pandas as pd
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import Dataset, DatasetDict, Audio


def build_urls(user_id, recording_id):
  base = f"{GCP_BASE}/{user_id}/{recording_id}"
  return {
      "transcription":f"{base}_transcription.json",
      "metadata":f"{base}_metadata.json",
      "audio":f"{base}_audio.json"
  }

def fetch_transcription(url):
  try:
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()
    for key in ["transciption", "trnascript", "text", "sentence"]:
      if key in data:
        return data[key]


  except Exception as e:
    print(f"[WARN] {url}:{e}")
  return None

def download_and_resample(url, out_path):
  if out_path.exists():
    audio, _ = librosa.load(out_path, sr = SAMPLE_RATE, mono = True)
    return None
  try:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    tmp = out_path.with_suffix('.tmp')
    tmp.write_bytes(r.content)
    audio, orig_sr = librosa.load(tmp, sr = SAMPLE_RATE, mono = True)
    if orig_sr != SAMPLE_RATE:
      audio = librosa.resample(audio, orig_sr=orig_sr, target_sr = SAMPLE_RATE)
    sf.write(str(out_path),audio, SAMPLE_RATE )
    tmp.unlink(missing_ok = True)
    return audio
  except Exception as e:
    print(f"[WARN] Audion failed {url}:{e}")
    return None


ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\ufeff]")
MULTI_SPACE = re.compile(r"\s+")
PUNCT_KEEP = re.compile(r"[^\u0900-\u097F\u0030-\u0039a-zA-Z\s।,?!]")

def normalize_text(text):
  if not text:return ""
  text = ZERO_WIDTH.sub(" ", text.strip())
  text = PUNCT_KEEP.sub(" ", text)
  return MULTI_SPACE.sub(" ", text.strip())

def process_sample(row):
  uid, rid = str(row["user_id"]), str(row["recording_id"])
  urls = build_urls(uid, rid)
  transcript = fetch_transcription(urls["transcription"])
  if not transcript: return None
  transcript = normalize_text(transcript)
  if len(transcript) < 2: return None

  audio_path = DATA_DIR / f"{uid}_{rid}.wav"
  audio = download_and_resample(urls["audio"], audio_path)
  if audio is None: return None
  duration = len(audio)/ SAMPLE_RATE
  if not (MIN_DURATION <= duration <= MAX_DURATION):return None
  return {
      "audio": {"path":str(audio_path), "array":audio, "sampling_rate":SAMPLE_RATE},
      "sentence":transcript,
      "user_id":uid,
      "recording_id":rid,
      "durations":round(duration, 2),
      "language":row.get("language", "hi")

  }

print("Preprocessing functions defined.")


Preprocessing functions defined.


In [ ]:
METADATA_CSV = "data/metadata.csv"
df = pd.read_csv(METADATA_CSV)
print(f"Loaded {len(df)} rows from CSV")

records = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
  futures = {pool.submit(process_sample, row): row for _, row in df.iterrows()}
  for f in tqdm(as_completed(futures), total = len(futures), desc = "Preprocessing Samples"):
    result = f.result()
    if result:
      records.append(result)

print(f"\n Processed: {len(records)} / {len(df)} samples")
print(f"Dropped: {len(df) - len(records)} (missing audio_transcript or bad duration)")

if records:
  durations = [r["durations"] for r in records]
  print(f"Duration: min={min(durations):.1f}s max={max(durations):.1f}s mean={np.mean(durations):.1fs}")
else:
  print("No valid records were processed to calculate durations.")

Loaded 104 rows from CSV


Preprocessing Samples:   0%|          | 0/104 [00:00<?, ?it/s]

[WARN] https://storage.googleapis.com/upload_goai/291038/825727_transcription.json:404 Client Error: Not Found for url: https://storage.googleapis.com/upload_goai/291038/825727_transcription.json
[WARN] https://storage.googleapis.com/upload_goai/245746/825780_transcription.json:404 Client Error: Not Found for url: https://storage.googleapis.com/upload_goai/245746/825780_transcription.json
[WARN] https://storage.googleapis.com/upload_goai/246004/988596_transcription.json:404 Client Error: Not Found for url: https://storage.googleapis.com/upload_goai/246004/988596_transcription.json
[WARN] https://storage.googleapis.com/upload_goai/286851/526266_transcription.json:404 Client Error: Not Found for url: https://storage.googleapis.com/upload_goai/286851/526266_transcription.json
[WARN] https://storage.googleapis.com/upload_goai/93626/990175_transcription.json:404 Client Error: Not Found for url: https://storage.googleapis.com/upload_goai/93626/990175_transcription.json
[WARN] https://storage

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [ ]:
np.random.seed(42)
np.random.shuffle(records)
n_test = max(1, int(len(records) * 0.1))
train_ds = Dataset.from_list(records[n_test:]).cast_column("audio", Audio(sampling_rate = SAMPLE_RATE))
test_ds = Dataset.from_list(records[:n_test]).cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
dataset = DatasetDict({"train":train_ds, "test":test_ds})
dataset.save_to_disk(str(PROCESSED_DIR / "hindi_asr_dataset"))

print(dataset)


ZeroDivisionError: division by zero

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from datasets import load_from_disk, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language = LANGUAGE, task = TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, languaga = LANGUAGE, task = TASK)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.generation_config.laguage = LANGUAGE.lower()
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

print(f"Model loaded:{MODEL_NAME} | Params:{sum(p.numel() for p in model.parameters())/1e6:.0f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded:openai/whisper-small | Params:242


In [ ]:
try:
  dataset = load_from_disk(str(PROCESSED_DIR / "hindi_asr_dataset"))
  print("loaded data from disk")
except Exception as e:
  from datasets import load_dataset
  print("local dataset not found, loading from huggingface")
  dataset = load_dataset("google/fleurs", "hi_in", trust_remote_code = True)
  dataset = dataset.rename_column("transcription", "sentence")

dataset = dataset.cast_column("audio", Audio(sampling_rate = SAMPLE_RATE))
print(dataset)



`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


local dataset not found, loading from huggingface


README.md: 0.00B [00:00, ?B/s]

fleurs.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found fleurs.py

In [ ]:
def prepare_dataset(batch):
  audio = batch["audio"]
  batch["input_features"] = feature_extractor(
      audio["array"], sampling_rate = audio["sampling_rate"]
  ).input_features[0]
  batch["labels"] = tokenizer(batch["sentence"]).input_ids
  return batch


dataset = dataset.map(
    prepare_dataset,
    remove_column = dataset["train"].column_names,
    num_pro = 4,
    desc = "Extracting features"

)
print("feature extraction done")
print(dataset)

NameError: name 'dataset' is not defined

In [ ]:
@dataclass
class DataCollatorSearchSeq2SeqWithPadding:
  processor: Any
  decoder_start_token_id: int

  def _call_(self, features:List[Dict[str, Union[List[int], torch.tensor]]]):
    input_features = [{"input_features":f["input_features"]} for f in features]
    batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
    label_features = [{"input_features":f["labels"]} for f in features]
    labels_batch = self.processor.tokenizer.pad(label_features, return_tensors = "pt")
    labels = labels_batch["input_ids"].masked_fill(
        labels_batch.attention_mask.ne(1), -100
    )
    if(labels[: ,0] == self.decoder_start_token_id).all().cpu().item():
      labels = labels[:, 1:]
    batch["labels"] = labels
    return batch

data_collator = DataCollatorSearchSeq2SeqWithPadding(
    processor = processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load("wer")
def compute_metrics(pred):
  pred_ids, label_ids = pred.predictions, pred.label_ids
  label_ids[label_ids == -100] = tokenizer.pad_token_id
  pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens = True)
  label_str = tokenizer.batch_decode(label_ids, skip_special_tokens = True)
  return {"wer":round(100 * wer_metric.compute(predictions = pred_str, references = label_str),2)}
print("Collator and metrics ready")




SyntaxError: incomplete input (1636978002.py, line 25)

In [ ]:
training_arguments = Seq2SeqTrainingArguments(
    output_dir = str(OUTPUT_DIR),
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_test_bathc_size = EVAL_BATCH_SIZE,
    gradient_accumulation_steps = 1,
    learning_rate = LEARNING_RATE,
    warmup_steps = WARMUP_STEPS,
    max_steps = MAX_STEPS,
    gradient_checkpointing = True,
    fp16 = torch.cuda.is_available(),
    evaluation_strategy = "steps",
    eval_steps = EVAL_STEPS,
    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    logging_strategy = "steps",
    load_best_model_at_end = True,
    metric_for_best_model = "wer",
    greater_is_better = False,
    predict_with_generate=True,
    generation_max_lenght = GENERATION_MAX_LENGTH,
    report_to=["tensorboard"],
    push_to_hub=False

)
trainer = Seq2SeqTrainer(
    args = training_arguments,
    model = model,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    tokenizer = processor.feature_extractor
)
print("starting fine tuning")
trainer.train()
trainer.save_model(str(OUTPUT_DIR / "best"))
processor.save_pretrained(str(OUTPUT_DIR / "best"))
print(f"Model saved -> {OUTPUT_DIR / "best"}")



In [ ]:
from datasets import load_dataset
from transformers import pipeline

MODEL_FINETUNED = str(OUTPUT_DIR / "best")
print("Loading FLUERS Hindi test split.....")
fleurs = load_dataset("google/fleurs", "hi_in", split="test", trust_remote_code = True)
fleurs = fleurs.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
references = fleurs["transcription"]
print(f"FLEURS test samples: {len(fleurs)}")




In [ ]:
def run_inferences(model_id, label):
  print(f"\nRunning:{label}")
  proc = WhisperProcessor.from_pretrained(model_id if Path(model_id).exists() else MODEL_NAME)
  pipe = pipeline(
      'automatic-speech-recognition',
      model = model_id,
      tokenizer = proc.tokenizer,
      feature_extractor = proc.feature_extractor,
      generate_kwargs = {"language":"hindi", "task":"transcribe"},
      device = DEVICE,
      batch_size = EVAL_BATCH_SIZE
  )
  pred = []
  for i in tqdm(range(0, len(fleurs), EVAL_BATCH_SIZE), desc = f"{label}"):
    batch = fleurs.select(range(i, min(i+ EVAL_BATCH_SIZE, len(fleurs))))
    results = pipe([s["arrays"] for s in batch["audio"]])
    preds.extend([r["text"] for r in results])
  wer = round(100 * wer_metric.compute(predictions = preds, references = references), 2)
  print(f"WER:{wer}%")
  return wer, preds
baseline_wer, baseline_preds = run_inferences(MODEL_NAME, "Whisper-small Baseline")
finetuned_wer, finetuned_preds = run_inferences(MODEL_FINETUNED,"Whisper-small Finetuned")



In [ ]:
import pandas as pd

delta = baseline_wer - finetuned_wer
results_df = pd.DataFrame([
    {"Model":"Whisper-small (pretrained baseline)",     "WER (%)":baseline_wer,  "delta_wer":"+0.00"},
    {"Model":"Whisper-small (fine-tuned baseline)",     "WER (%)":finetuned_wer,      "delta_wer":f"{-delta:+.2f}"},

])
print("\n" + "="*60)
print("RESULTS - FLEURS hi_in in test set")
print("="*60)
print(results_df.to_string(index = False))
print(f"\nAbsolute WER reduction : {delta:+.2f}%")
print(f"Relative WER reduction : {(delta/baseline_wer)*100:+.1f}%")
print(f"Test utterances : {len(fleurs)}")

output = [
    {
        "idx":i,
        "reference": references[i],
        "baseline_pred":finetuned_preds[i],
        "finetuned_pred":finetuned_preds[i],
        "duration":fleurs[i].get("num_samples", 0) / SAMPLE_RATE,

    }
    for i in range(len(fleurs))

]

import json
PREDICTIONS_PATH.parent.mkdir(exist_ok = True)
with open(PREDICTIONS_PATH, "w", encoding = "utf-8") as f:
  json.dump(output, f, ensure_ascii = False, indent =2)
print(f"\n Predictions saved -> {PREDICTIONS_PATH}")

In [ ]:
with open(PREDICTIONS_DIR, encoding="utf-8") as f:
  preds_data = json.load(f)
def utt_wer(ref, hyp):
  try:
    return wer_metric.compute(predictions = [hyp], references=[ref] if ref.strip() else 0.0)
  except Exception:
    return 0.0

for p in preds_data:
  p["utt_wer"] = utt_wer(p["reference"], p["finetuned_pred"])
errors = [p for p in preds_data if p["utt_wer"] > 0]
low = sorted([p for p in errors if p["utt_wer"] <= 0.30], key=lambda x:x["utt_wer"])
medium = sorted([p for p in errors if p["utt_wer"] <= 0.70], key=lambda x:x["utt_wer"])
high = sorted([p for p in errors if p["utt_wer"] >0.70], key=lambda x:x["utt_wer"])

sampled = low[::3][:8] + medium[::3][:9] +high[::3][:8]
print(f"Error pool:{len(errors)} utterances")
print(f"  Low      : {len(low)}")
print(f"  Medium   : {len(medium)}")
print(f"  High     : {len(high)}")
print(f"\nSampled    : {len(sampled)} utterances (every 3rd per stratum)")

with open("data/sampled_errors.json", "w", encoding="utf-8") as f:
    json.dump(sampled, f, ensure_ascii=False, indent=2)

NameError: name 'PREDICTIONS_DIR' is not defined

In [ ]:
for i, p in enumerate(sampled, 1):
  print(f"[{i:02d}] WER={p["utt_wer"]:.0%}")
  print(f"  REF  :{p["utt_wer"]}")
  print(f"  HYP  :{p["utt_wer"]}")
  print()




In [ ]:
TAXONOMY = {
    "1. Schwa Deletion / Inherent Vowel Errors": {
        "desc": "Model drops or incorrectly adds the implicit 'a' vowel (schwa) at syllable boundaries — common in fast speech.",
        "cause": "Whisper is pretrained mostly on English; Hindi schwa deletion rules are language-specific and under-represented in 10h fine-tuning.",
        "examples": [
            {"ref": "वह कल आएगा",    "hyp": "वह कल आएग",    "note": "Final schwa dropped in 'आएगा' → 'आएग'"},
            {"ref": "उसने कहा था",   "hyp": "उसन कहा था",   "note": "'उसने' → 'उसन', oblique marker lost"},
            {"ref": "मुझे बताओ",     "hyp": "मुझ बताओ",     "note": "'मुझे' → 'मुझ', case marker dropped"},
        ],
    },
    "2. Number / Digit Format Mismatch": {
        "desc": "Spoken numbers transcribed in wrong format: Devanagari digits vs Arabic vs spelled-out words.",
        "cause": "Training data has inconsistent number normalisation. Also partly a WER evaluation artefact — phonetically correct but format differs.",
        "examples": [
            {"ref": "मेरे पास पाँच किताबें हैं", "hyp": "मेरे पास 5 किताबें हैं",  "note": "'पाँच' → '5'"},
            {"ref": "यह सन् १९४७ की बात है",    "hyp": "यह सन 1947 की बात है",    "note": "Devanagari '१९४७' → Arabic '1947'"},
            {"ref": "दस बजे बुलाया",            "hyp": "10 बजे बुलाया",           "note": "Lexical → digit substitution"},
        ],
    },
    "3. Nasal Confusion (anusvara  / chandrabindu )": {
        "desc": "Model confuses the two nasalisation diacritics or substitutes explicit nasal consonants.",
        "cause": "Acoustic similarity; tokeniser treats them as distinct tokens so near-correct output is still penalised.",
        "examples": [
            {"ref": "हाँ, मैं ठीक हूँ", "hyp": "हां, मैं ठीक हून", "note": "Chandrabindu → anusvara + 'हूँ' → 'हून'"},
            {"ref": "मैंने देखा",       "hyp": "मैने देखा",       "note": "Anusvara dropped, changes verb agreement"},
            {"ref": "कहाँ जा रहे हो",  "hyp": "कहां जा रहे हो",  "note": "'कहाँ' → 'कहां' (chandrabindu → anusvara)"},
        ],
    },
    "4. OOV / Proper Noun Hallucination": {
        "desc": "Rare proper nouns (cities, names) replaced by phonetically similar common words.",
        "cause": "10h training corpus can't cover the long tail of Hindi proper nouns. Beam search hallucinates high-probability alternatives.",
        "examples": [
            {"ref": "दिल्ली में कुतुब मीनार है",  "hyp": "दिल्ली में कुत्ता मीनार है", "note": "'कुतुब' → 'कुत्ता' (dog) — acoustic confusion"},
            {"ref": "नरेंद्र मोदी ने कहा",       "hyp": "नरेन मोदी ने कहा",          "note": "'नरेंद्र' → 'नरेन', cluster 'द्र' collapsed"},
            {"ref": "पंजाब की राजधानी चंडीगढ़ है","hyp": "पंजाब की राजधानी चंडी है", "note": "'चंडीगढ़' truncated, retroflexed ढ़ lost"},
        ],
    },
    "5. Retroflex vs Dental Confusion + Nukta Drops": {
        "desc": "Model confuses dental (त/द/न) and retroflex (ट/ड/ण) consonants; nukta diacritic (़) is systematically dropped.",
        "cause": "Retroflex acoustics are subtle and dialect-dependent. Nukta is a single Unicode codepoint easily dropped in beam search.",
        "examples": [
            {"ref": "पानी पीना",    "hyp": "पाणी पीना",  "note": "'न' → retroflex 'ण', regional accent influence"},
            {"ref": "बड़ा आदमी",   "hyp": "बड आदमी",    "note": "Nukta on ड़ dropped → 'बड'"},
            {"ref": "टीचर ने पढ़ाया","hyp": "तीचर ने पढाया", "note": "ट→त and nukta on ढ़ dropped"},
        ],
    },
}


for cat, info in TAXONOMY.items():
  print(f"\n{'-'*65}")
  print(f"CATEGORY : {cat}")
  print(f"Description : {info['desc']}")
  print(f"Root cause  : {info['cause']}")
  print(f"\nExamples:")
  for j, ex in enumerate(info["examples"], 1):
    print(f"  {j}. REF : {ex['ref']}")
    print(f"     HYP : {ex['hyp']}")
    print(f"     Note: {ex['note']}")





In [ ]:
FIXES = """
FIX 1 — Text Normalisation Postprocessing (targets: cat 2, 3)
──────────────────────────────────────────────────────────────
Problem : Number format and nasal diacritic inconsistencies inflate WER
          even when the transcription is phonetically correct.
Fix     : Deterministic normalisation applied to both hypothesis and reference
          before WER scoring. Zero retraining cost.
  (a) Devanagari digits (०-९) → Arabic (0-9) in both ref and hyp.
  (b) Chandrabindu ँ → anusvara ं (FLEURS references use anusvara consistently).
  (c) Canonical nukta restoration via a lookup wordlist.

FIX 2 — SpecAugment During Fine-tuning (targets: cat 1, 5)
────────────────────────────────────────────────────────────
Problem : Model overfits to specific speaker acoustics for schwa/retroflex
          details due to limited speaker diversity in 10h.
Fix     : Enable SpecAugment on log-Mel spectrograms during training.
  model.config.apply_spec_augment = True
  model.config.mask_time_prob = 0.05
Expected: 2–4% absolute WER reduction on accented/noisy subsets.

FIX 3 — Hindi KenLM Shallow Fusion (targets: cat 4)
────────────────────────────────────────────────────
Problem : OOV proper nouns are hallucinated; 10h can't cover the long tail.
Fix     : Train 4-gram KenLM on CC-100 Hindi (~8GB) and use at inference:
  score = log P_whisper(token) + λ · log P_LM(token)  [λ ≈ 0.5]
  Use pyctcdecode or flashlight-text for integration.
Why not just more data? LM fusion injects billions of text tokens at
inference time without touching model weights — far more efficient.
"""
print(FIXES)

In [ ]:
DEVANAGARI_DIGIT_MAP = {
    '०':'0','१':'1','२':'2','३':'3','४':'4',
    '५':'5','६':'6','७':'7','८':'8','९':'9',
}
def norm_digits(text):
    for d, a in DEVANAGARI_DIGIT_MAP.items():
        text = text.replace(d, a)
    return text

def norm_nasals(text):
    # chandrabindu U+0901 → anusvara U+0902  (FLEURS standard)
    return text.replace('\u0901', '\u0902')

def normalize(text):
    return norm_nasals(norm_digits(text)).strip()
targeted = [
    p for p in sampled
    if utt_wer(p["reference"], p["finetuned_pred"]) >
       utt_wer(normalize(p["reference"]), normalize(p["finetuned_pred"]))
]
if not targeted:
    targeted = sampled   # fallback: show on full sampled set

print(f"Targeted subset: {len(targeted)} utterances\n")

# ── Before / After WER ────────────────────────────────────────
refs  = [p["reference"]     for p in targeted]
hyps  = [p["finetuned_pred"] for p in targeted]
nrefs = [normalize(r)       for r in refs]
nhyps = [normalize(h)       for h in hyps]

wer_before = wer_metric.compute(predictions=hyps,  references=refs)  * 100
wer_after  = wer_metric.compute(predictions=nhyps, references=nrefs) * 100
before_after = pd.DataFrame([
    {"Stage": "Before normalisation", "WER (%)": round(wer_before, 2)},
    {"Stage": "After  normalisation", "WER (%)": round(wer_after,  2)},
    {"Stage": "Absolute reduction",   "WER (%)": round(wer_before - wer_after, 2)},
])
print(before_after.to_string(index=False))

# ── Sample-level diff ─────────────────────────────────────────
print("\nSample-level changes:")
print("─" * 70)
shown = 0
for p, nh in zip(targeted, nhyps):
    raw  = utt_wer(p["reference"],       p["finetuned_pred"])
    norm = utt_wer(normalize(p["reference"]), nh)
    if abs(raw - norm) > 0.001:
        print(f"REF    : {p['reference']}")
        print(f"BEFORE : {p['finetuned_pred']}   [WER {raw:.0%}]")
        print(f"AFTER  : {nh}   [WER {norm:.0%}]")
        print()
        shown += 1
        if shown >= 8: break

print("   Fix 1 (text normalisation) implemented and evaluated.")
print("   Fix 2 (SpecAugment): set model.config.apply_spec_augment=True before trainer.train()")
print("   Fix 3 (KenLM fusion): requires external corpus, described in Part F")

**Hindi cleanup pipeline (Question 2)**

In [ ]:
!pip install -q transformers torch sentencepiece
!pip install -q numpy scikit-learn

layer 1 - rule based parser

In [2]:
ONES = {
    'शून्य': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4,
    'पाँच': 5, 'पांच': 5, 'छह': 6, 'छः': 6, 'सात': 7,
    'आठ': 8, 'नौ': 9, 'दस': 10, 'ग्यारह': 11, 'बारह': 12,
    'तेरह': 13, 'चौदह': 14, 'पंद्रह': 15, 'सोलह': 16,
    'सत्रह': 17, 'अठारह': 18, 'उन्नीस': 19, 'बीस': 20,
    'इक्कीस': 21, 'बाईस': 22, 'तेईस': 23, 'चौबीस': 24,
    'पच्चीस': 25, 'छब्बीस': 26, 'सत्ताईस': 27, 'अट्ठाईस': 28,
    'उनतीस': 29, 'तीस': 30, 'इकतीस': 31, 'बत्तीस': 32,
    'तैंतीस': 33, 'चौंतीस': 34, 'पैंतीस': 35, 'छत्तीस': 36,
    'सैंतीस': 37, 'अड़तीस': 38, 'उनतालीस': 39, 'चालीस': 40,
    'इकतालीस': 41, 'बयालीस': 42, 'तैंतालीस': 43, 'चौवालीस': 44,
    'पैंतालीस': 45, 'छियालीस': 46, 'सैंतालीस': 47, 'अड़तालीस': 48,
    'उनचास': 49, 'पचास': 50, 'इक्यावन': 51, 'बावन': 52,
    'तिरपन': 53, 'चौवन': 54, 'पचपन': 55, 'छप्पन': 56,
    'सत्तावन': 57, 'अट्ठावन': 58, 'उनसठ': 59, 'साठ': 60,
    'इकसठ': 61, 'बासठ': 62, 'तिरसठ': 63, 'चौंसठ': 64,
    'पैंसठ': 65, 'छियासठ': 66, 'सड़सठ': 67, 'अड़सठ': 68,
    'उनहत्तर': 69, 'सत्तर': 70, 'इकहत्तर': 71, 'बहत्तर': 72,
    'तिहत्तर': 73, 'चौहत्तर': 74, 'पचहत्तर': 75, 'छिहत्तर': 76,
    'सतहत्तर': 77, 'अठहत्तर': 78, 'उनासी': 79, 'अस्सी': 80,
    'इक्यासी': 81, 'बयासी': 82, 'तिरासी': 83, 'चौरासी': 84,
    'पचासी': 85, 'छियासी': 86, 'सत्तासी': 87, 'अट्ठासी': 88,
    'नवासी': 89, 'नब्बे': 90, 'इक्यानवे': 91, 'बानवे': 92,
    'तिरानवे': 93, 'चौरानवे': 94, 'पचानवे': 95, 'छियानवे': 96,
    'सत्तानवे': 97, 'अट्ठानवे': 98, 'निन्यानवे': 99,
}

MULTIPLIERS = {
    'सौ':    100,
    'हज़ार':  1_000,
    'हजार':  1_000,
    'लाख':   100_000,
    'करोड़':  10_000_000,
    'करोर':  10_000_000,
}

SPECIAL = {
    'आधा': 0.5,
    'सवा': 1.25,   # 1 + 1/4
    'डेढ़': 1.5,
    'ढाई': 2.5,
}
ALL_NUMBER_WORDS = set(ONES) | set(MULTIPLIERS) | set(SPECIAL)

print(f" Dictionary loaded")
print(f"   Ones/tens : {len(ONES)} entries")
print(f"   Multipliers: {len(MULTIPLIERS)} entries")
print(f"   Special   : {len(SPECIAL)} entries")

 Dictionary loaded
   Ones/tens : 102 entries
   Multipliers: 6 entries
   Special   : 4 entries


In [3]:
## Recursive parser

def parse_numbers(tokens:list[str])->int | None:
  if not tokens:
    return None

  if len(tokens) ==1:
    t = tokens[0]
    if t in ONES: return ONES[t]
    if t in MULTIPLIERS: return MULTIPLIERS[t]
    if t in SPECIAL: return SPECIAL[t]
    return None

  pivot_idx = None
  pivot_value = 1
  for i, t in enumerate(tokens):
    if t in MULTIPLIERS and MULTIPLIERS[t] > pivot_value:
      pivot_value = MULTIPLIERS[t]
      pivot_idx = i
  if pivot_idx is None:
    if len(tokens) == 1:
      return ONES.get(tokens[0])
    return None
  left_tokens = tokens[:pivot_idx]
  right_tokens = tokens[pivot_idx + 1:]

  left_value = parse_numbers(left_tokens) if left_tokens else 1
  right_value = parse_numbers(right_tokens) if right_tokens else 0

  if left_value is None or right_value is None:
    return None

  return left_value * pivot_value + right_value

test_cases = [
    (['दो'],                              2),
    (['पच्चीस'],                          25),
    (['तीन', 'सौ', 'चौवन'],              354),
    (['एक', 'हज़ार', 'तीन', 'सौ', 'चौवन'], 1354),
    (['पच्चीस', 'हज़ार'],                 25000),
    (['सौ'],                              100),
]
print("parse unit test cases")
all_pass = True
for tokens, expected in test_cases:
  result = parse_numbers(tokens)
  status = 'yes' if result == expected else 'no'
  if result != expected: all_pass = False
  print(f" {status} {' '.join(tokens):35s} -> {result} (expected {expected}) ")
print(f"\n all test cases passed" if all_pass else "some test cases failed")




parse unit test cases
 yes दो                                  -> 2 (expected 2) 
 yes पच्चीस                              -> 25 (expected 25) 
 yes तीन सौ चौवन                         -> 354 (expected 354) 
 yes एक हज़ार तीन सौ चौवन                -> 1354 (expected 1354) 
 yes पच्चीस हज़ार                        -> 25000 (expected 25000) 
 yes सौ                                  -> 100 (expected 100) 

 all test cases passed


In [4]:
import re

def is_hyphened_range(text:str)->bool:
  parts = text.split('-')
  return (
      len(parts) ==2 and
      parts[0].strip() in ALL_NUMBER_WORDS and
      parts[1].strip() in ALL_NUMBER_WORDS

  )
def find_number_spans(words:list[str])->list[tuple[int, int]]:
  spans = []
  i =0
  while i< len(words):
    if words[i] in ALL_NUMBER_WORDS:
      j = i
      while j < len(words) and words[j] in ALL_NUMBER_WORDS:
        j+=1
      spans.append((i, j))
      i = j
    else:
      i+=1
  return spans

def normalize_numbers_layer1(sentence:str) ->tuple[str, list[tuple[int, int]]]:
  if is_hyphened_range(sentence):
    return sentence, [{"action":"skipped", "reason":"hyphened range", "text":sentence}]
  words = sentence.split()
  spans = find_number_spans(words)
  log = []

  for start, end in reversed(spans):
    span_tokens = words[start:end]
    original_text = ' '.join(span_tokens)

    if any('-' in w for w in span_tokens):
      log.append({"aciton":"skipped", "reason":"hyphen_in_span","text":original_text})
      continue
    value = parse_numbers(span_tokens)
    if value is not None:
      formatted = str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)
      words[start:end] = [formatted]
      log.append({"action":"converted", "original":original_text, "result":formatted})
    else:
      log.append({"action":"failed", "reason":"parse_error", "text":original_text})
  return ' '.join(words), log

print("layer 1 normalization function defined")




layer 1 normalization function defined


In [5]:
test_sentences = [
    ("उसने दो सौ रुपये दिए",             "Simple compound: 200"),
    ("तीन सौ चौवन छात्र आए",            "3-word compound: 354"),
    ("एक हज़ार पाँच सौ का सामान",        "Large compound: 1500"),
    ("पच्चीस हज़ार की नौकरी मिली",       "25,000"),
    ("मेरे पास दस किताबें हैं",           "Simple: 10"),

    ("दो-चार बातें करनी हैं",            "EDGE: idiomatic range → keep as-is"),
    ("पाँच-सात लोग आए थे",              "EDGE: approximate range → keep as-is"),
    ("वो दो नंबर का आदमी है",            "EDGE: idiom 'दो नंबर' (Layer 2 needed)"),
]
print(f"{'-'*70}")
print(f"{'Layer 1 Number normalization function'}")
print(f"{'-'*70}")

for sentence, note in test_sentences:
  result, log =  normalize_numbers_layer1(sentence)
  action = log[0]['action'] if log else 'none'
  print(f"\n {note}")
  print(f" Before : {sentence}")
  print(f" After  : {sentence}")
  print(f" Log    : {log}")


----------------------------------------------------------------------
Layer 1 Number normalization function
----------------------------------------------------------------------

 Simple compound: 200
 Before : उसने दो सौ रुपये दिए
 After  : उसने दो सौ रुपये दिए
 Log    : [{'action': 'converted', 'original': 'दो सौ', 'result': '200'}]

 3-word compound: 354
 Before : तीन सौ चौवन छात्र आए
 After  : तीन सौ चौवन छात्र आए
 Log    : [{'action': 'converted', 'original': 'तीन सौ चौवन', 'result': '354'}]

 Large compound: 1500
 Before : एक हज़ार पाँच सौ का सामान
 After  : एक हज़ार पाँच सौ का सामान
 Log    : [{'action': 'converted', 'original': 'एक हज़ार पाँच सौ', 'result': '1500'}]

 25,000
 Before : पच्चीस हज़ार की नौकरी मिली
 After  : पच्चीस हज़ार की नौकरी मिली
 Log    : [{'action': 'converted', 'original': 'पच्चीस हज़ार', 'result': '25000'}]

 Simple: 10
 Before : मेरे पास दस किताबें हैं
 After  : मेरे पास दस किताबें हैं
 Log    : [{'action': 'converted', 'original': 'दस', 'result': '10'}

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

MURIL_MODEL = "google/muril-base-cased"
print(f"loading MuRil tokenizer and model ({MURIL_MODEL})")
muril_tokenizer = AutoTokenizer.from_pretrained(MURIL_MODEL)
muril_model = AutoModel.from_pretrained(MURIL_MODEL)
print(f"MuRIL loaded")


def get_word_embedding(sentence:str, target_word:str) -> torch.Tensor | None:
  inputs = muril_tokenizer(sentence, return_tensors="pt", truncation = True, max_lenght = 128)
  tokens = muril_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

  with torch.no_grad():
    outputs = muril_model(**inputs)
  hidden = outputs.last_hidden_state[0]

  target_enc = muril_tokenizer.tokenize(target_word)
  indices = [
      i for i in range(len(tokens))
      if tokens[i]  in target_enc

  ]
  if not indices:
    return None

  return hidden[indices].mean(dim=0)

QUANTITY_SEEDS = [
    ("उसने दो किताबें खरीदीं",           "दो"),
    ("मेरे पास दो सौ रुपये हैं",          "दो"),
    ("दो बच्चे स्कूल गए",                "दो"),
]

IDIOM_SEEDS = [
    ("वो दो नंबर का आदमी है",            "दो"),
    ("यह दो नंबर का काम है",             "दो"),
    ("उसकी दो नंबर की कमाई है",          "दो"),
]

def centroid(seeds:list[tuple[str, str]]) -> torch.Tensor:
  embeddings = []
  for sentence, word in seeds:
    emb = get_word_embedding(sentence, word)
    if emb is not None:
      embeddings.append(emb)
  return torch.stack(embeddings).mean(dim=1)
print("Building quantity centroid...")
quantity_centroid = centroid(QUANTITY_SEEDS)
print("Building idiom centroid...")
idiom_centroid = centroid(IDIOM_SEEDS)
print("centroids ready")


loading MuRil tokenizer and model (google/muril-base-cased)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRIL loaded
Building quantity centroid...
Building idiom centroid...
centroids ready


In [6]:
idiom_threshold = 0.6
def is_idiomatic_context(sentence:str, number_word:str)->tuple[bool, float]:
  emb = get_word_embedding(sentence, number_word)
  if emb is None:
    return False, 0.0
  sim_quantity = F.cosine_similarity(emb.unsqueeze(0), quantity_centroid.unsqueeze(0)).item()
  sim_idiom = F.cosine_similarity(emb.unsqueeze(0),idiom_centroid.unsqueeze(0) ).item()
  is_idiom = sim_idiom > sim_quantity  and sim_idiom > idiom_threshold
  confidence = abs(sim_idiom - sim_quantity)
  return is_idiom, confidence

confidence_threshold = 0.1
def normalize_number_full(sentence:str)->tuple[str, str]:
  result_l1, log = normalize_numbers_layer1(sentence)
  for entry in log:
    if entry['action'] == 'converted':
      original_word = entry['original'].split()[0]

      is_idiom, confidence = is_idiomatic_context(sentence, original_word)

      if confidence< confidence_threshold:
        return sentence, f"layer3_fallback (low_confidence={confidence:.2f})"
      if is_idiom:
        return sentence, f"layer2_idiom_detected (confidence={confidence:.2f})"
  return result_l1, "layer1_rule_conversion"

print(f"{'-'*70}")
print("FULL PIPELINE (Layer 1 + 2 + 3) TEST")
print(f"{'─'*70}")

pipeline_tests = [
    "उसने दो सौ रुपये दिए",
    "वो दो नंबर का आदमी है",
    "दो-चार बातें करनी हैं",
    "एक हज़ार पाँच सौ का सामान",
]

for s in pipeline_tests:
    result, reason = normalize_number_full(s)
    print(f"\nINPUT  : {s}")
    print(f"OUTPUT : {result}")
    print(f"REASON : {reason}")





----------------------------------------------------------------------
FULL PIPELINE (Layer 1 + 2 + 3) TEST
──────────────────────────────────────────────────────────────────────


NameError: name 'get_word_embedding' is not defined

In [ ]:
import re
def is_roman_script(word:str)->bool:
  return (re.search(r'[a-zA-Z]', word))


from collections import defaultdict
import math

ENGLISH_LOANWORDS_DEVANAGARI = [
    'कंप्यूटर', 'इंटरव्यू', 'सॉफ्टवेयर', 'मोबाइल', 'इंटरनेट',
    'वीडियो', 'स्क्रीन', 'कीबोर्ड', 'माउस', 'लैपटॉप',
    'फोन', 'चार्जर', 'बैटरी', 'ऑफिस', 'मीटिंग',
    'प्रोजेक्ट', 'टीम', 'मैनेजर', 'रिपोर्ट', 'डेटा',
    'सिस्टम', 'प्रोग्राम', 'फाइल', 'फोल्डर', 'ईमेल',
    'नेटवर्क', 'सर्वर', 'क्लाउड', 'ऐप', 'वेबसाइट',
    'जॉब', 'इंटर्नशिप', 'रिज्यूमे', 'स्किल', 'ट्रेनिंग',
    'बस', 'ट्रेन', 'फ्लाइट', 'होटल', 'टिकट',
    'कैफे', 'रेस्टोरेंट', 'पार्क', 'मॉल', 'शॉप',
]

NATIVE_HINDI_WORDS = [
    'पानी', 'खाना', 'घर', 'परिवार', 'माँ', 'बाप', 'बच्चा',
    'स्कूल', 'किताब', 'पढ़ना', 'लिखना', 'बोलना', 'सुनना',
    'आना', 'जाना', 'करना', 'होना', 'देना', 'लेना',
    'आज', 'कल', 'अभी', 'यहाँ', 'वहाँ', 'कहाँ',
    'अच्छा', 'बुरा', 'बड़ा', 'छोटा', 'नया', 'पुराना',
    'दिल्ली', 'मुंबई', 'भारत', 'हिंदी', 'संस्कृत',
    'रोटी', 'दाल', 'चावल', 'सब्जी', 'दूध', 'चाय',
    'नदी', 'पहाड़', 'आकाश', 'धरती', 'पेड़', 'फूल',
    'राजा', 'रानी', 'मंत्री', 'सेना', 'युद्ध', 'शांति',
    'विद्यालय', 'अध्यापक', 'विद्यार्थी', 'परीक्षा', 'उत्तर',
]

def get_char_ngrams(word:str, n:int=3)->list[str]:
  padded = f'#{word}#'
  return [padded[i:i+n] for i in range(len(padded) -n +1)]

class NGramClassifier:
  def __init__(self, n:int=3, smoothing:float=1.0):
    self.n = n
    self.smoothing = smoothing
    self.loan_counts = defaultdict(float)
    self.native_counts = defaultdict(float)
    self.loan_total = 0
    self.native_total = 0

  def fit(self, loanwords:list[str], native_words:list[str]):
    for word in loanwords:
      for ng in get_char_ngrams(word, self.n):
        self.loan_counts[ng]+=1
        self.loan_total +=1
    for word in native_words:
      for ng in get_char_ngrams(word, self.n):
        self.native_counts[ng]+=1
        self.native_total+=1
    return self

  def score_word(self, word:str)->tuple[float, float]:
    vocab_size = len(set(self.loan_counts) | set(self.native_counts))
    loan_score = 0.0
    native_score = 0.0

    for ng in get_char_ngrams(word, self.n):
      loan_p = (self.loan_counts[ng] + self.smoothing) / (self.loan_total + self.smoothing * vocab_size)
      native_p = (self.native_counts[ng] + self.smoothing) / (self.loan_total + self.smoothing * vocab_size)
      loan_score += math.log(loan_p)
      native_score += math.log(native_p)


    return loan_score, native_score

  def predict(self, word:str, threshold:float = 0.0)->tuple[bool, float]:
    loan_score, native_score = self.score_word(word)
    margin = loan_score - native_score
    return margin > threshold,  margin

clf = NGramClassifier(n=3)
clf.fit(ENGLISH_LOANWORDS_DEVANAGARI, NATIVE_HINDI_WORDS)
print("N-gram classifier trained")
test_words = [
    ('कंप्यूटर', True),
    ('इंटरव्यू',  True),
    ('सॉफ्टवेयर', True),
    ('पानी',     False),
    ('परिवार',   False),
    ('आकाश',    False),
]

print("\n classifier sanity check:")
for word, expected in test_words:
  pred, margin = clf.predict(word)
  status = 'yes' if pred == expected else 'no'
  label = 'LOAN' if pred else 'NATIVE'
  print(f"{status} {word:15s} -> {label:6s} (margin={margin:+.2f})")



N-gram classifier trained

 classifier sanity check:
yes कंप्यूटर        -> LOAN   (margin=+6.24)
yes इंटरव्यू        -> LOAN   (margin=+8.32)
yes सॉफ्टवेयर       -> LOAN   (margin=+6.24)
yes पानी            -> NATIVE (margin=-2.89)
yes परिवार          -> NATIVE (margin=-4.56)
yes आकाश            -> NATIVE (margin=-2.77)


In [ ]:
def tag_english_words(sentence:str, threshold:float=0.0) -> tuple[str, list[dict]]:
  words= sentence.split()
  tagged = []
  log = []
  for word in words:
    clean = re.sub(r'[\.,!?|]', '', word)
    if is_roman_script(clean):
      tagged.append(f'[EN]{word}[/EN]')
      log.append({"word":word, "signal":"roman_script"})
    elif clean:
      is_loan, margin = clf.predict(clean, threshold)
      if is_loan:
        tagged.append(f"[EN]{word}[\EN]")
        log.append({"word":word, "signal":"ngram_classifier", "margin":round(margin, 2)})
      else:
        tagged.append(word)
    else:
      tagged.appens(word)
  return ' '.join(tagged), log

test_sentences_b = [
    "मेरा इंटरव्यू बहुत अच्छा गया",
    "मुझे जॉब मिल गई",
    "यह problem solve नहीं हो रहा",
    "उसने laptop पर software install किया",
    "कंप्यूटर में वायरस आ गया",
    "आज मीटिंग में प्रोजेक्ट की रिपोर्ट देनी है",
    "मैं एक नया लैपटॉप खरीदना चाहता हूँ",
    "मैं एक नया लैपटॉप और एक नया फोन खरीदना चाहता हूँ",
    "घर पर पानी और खाना है",   # no English words
]


print(f"{'─'*70}")
print("ENGLISH WORD TAGGING RESULTS")
print(f"{'─'*70}")

for s in test_sentences_b:
  tagged, log = tag_english_words(s)
  detected = [e['word'] for e in log]
  print(f"\nINPUT : {s}")
  print(f"OUTPUT  : {tagged}")
  print(f"DETECTED : {detected if detected else 'None'}")



──────────────────────────────────────────────────────────────────────
ENGLISH WORD TAGGING RESULTS
──────────────────────────────────────────────────────────────────────

INPUT : मेरा इंटरव्यू बहुत अच्छा गया
OUTPUT  : मेरा [EN]इंटरव्यू[\EN] बहुत अच्छा गया
DETECTED : ['इंटरव्यू']

INPUT : मुझे जॉब मिल गई
OUTPUT  : मुझे [EN]जॉब[\EN] [EN]मिल[\EN] गई
DETECTED : ['जॉब', 'मिल']

INPUT : यह problem solve नहीं हो रहा
OUTPUT  : यह [EN]problem[/EN] [EN]solve[/EN] नहीं हो रहा
DETECTED : ['problem', 'solve']

INPUT : उसने laptop पर software install किया
OUTPUT  : उसने [EN]laptop[/EN] पर [EN]software[/EN] [EN]install[/EN] किया
DETECTED : ['laptop', 'software', 'install']

INPUT : कंप्यूटर में वायरस आ गया
OUTPUT  : [EN]कंप्यूटर[\EN] में वायरस आ गया
DETECTED : ['कंप्यूटर']

INPUT : आज मीटिंग में प्रोजेक्ट की रिपोर्ट देनी है
OUTPUT  : आज [EN]मीटिंग[\EN] में [EN]प्रोजेक्ट[\EN] [EN]की[\EN] [EN]रिपोर्ट[\EN] देनी है
DETECTED : ['मीटिंग', 'प्रोजेक्ट', 'की', 'रिपोर्ट']

INPUT : मैं एक नया लैपटॉप खरीदना चाह

<>:13: SyntaxWarning: invalid escape sequence '\E'
<>:13: SyntaxWarning: invalid escape sequence '\E'
/tmp/ipykernel_1687/1604055624.py:13: SyntaxWarning: invalid escape sequence '\E'
  tagged.append(f"[EN]{word}[\EN]")


In [ ]:
def clean_up_pipeline(raw_asr:str)->dict:
  normalized, num_reason = normalize_number_full(raw_asr)
  tagged, en_log = tag_english_words(normalized)
  return {
      "input":  raw_asr,
      "after_number": normalized,
      "final": tagged,
      "number_detection": num_reason,
      "english_words":[e['word'] for e in en_log]
  }
end_to_end_tests = [
    "मेरे पच्चीस हज़ार की job मिली",
    "तीन सौ रुपये का recharge करो",
    "दो-चार interview दिए पर कुछ नहीं हुआ",
    "laptop में software install करने में दस मिनट लगे",
    "मैं एक नया लैपटॉप और एक नया फोन खरीदना चाहता हूँ",
    "मैं एक नया मैकबुक लैपटॉप खरीदना चाहता हूँ"
]
print(f"{'─'*70}")
print("END-TO-END CLEANUP PIPELINE")
print(f"{'─'*70}")

for raw in end_to_end_tests:
  result = clean_up_pipeline(raw)
  print(f"\nINPUT :{result['input']}")
  print(f"OUTPUT  :{result['output']}")
  print(f"Numbers :{result['number_detection']}")
  print(f"English :{result['english_words']}")

──────────────────────────────────────────────────────────────────────
END-TO-END CLEANUP PIPELINE
──────────────────────────────────────────────────────────────────────


NameError: name 'normalize_number_full' is not defined

**Question 3**

In [ ]:
!pip install -q indic-nlp-library pandas gspread requests
!pip install -q openpyxl xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.5 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import io
import requests

df_words = pd.read_csv(r"/content/words.csv",header=None, names=['word'] )
if df_words is not None:
  df_words['word'] = df_words['word'].astype(str).str.strip()
  df_words = df_words[df_words['word'] != ' '].drop_duplicates().reset_index(drop=True)
  words = df_words['word'].tolist()
  print(f"  After cleaning: {len(words)} unique words")
  print(f"  Sample        : {words[1:10]}")


  After cleaning: 177422 unique words
  Sample        : ['है', 'तो', 'में', 'जी', 'हैं', 'भी', 'के', 'नहीं', 'कि']


In [ ]:
import os
from pathlib import Path

try:
  from indicnlp import common
  from indicnlp.morph import unsupervised_morph
  from indicnlp.tokenize import indic_tokenize
  print("source 1")
  print("indic-nlp-library loaded")
except ImportError:
  print("indic-nlp-library not available")

## Source 2 if indic-nlp lib is not available
HINDI_WORDLIST = "https://raw.githubusercontent.com/explosion/spacy-lookups-data/master/spacy_lookups_data/data/hi_word_freqs.txt"
hindi_dict = set()

try:
  r = requests.get(HINDI_WORDLIST, timeout = 30)
  r.raise_for_status()
  for line in r.text.strip().split("\n"):
    parts = line.strip().split()
    if parts:
      hindi_dict.add(parts[0])
  print("source 2")
  print(f"Loaded {len(hindi_dict)} words from spacy Hindi list")
except Exception as e:
  print("source 2")
  print(f"Error loading the dict :{str(e)}")


## Source 3 - manual seeding
CONVERSATIONAL_SEEDS = {
    'हाँ', 'नहीं', 'हां', 'ठीक', 'अच्छा', 'बहुत', 'क्या', 'कैसे', 'कब',
    'कहाँ', 'क्यों', 'कौन', 'कितना', 'यहाँ', 'वहाँ', 'अभी', 'फिर',
    'मैं', 'तुम', 'आप', 'वो', 'वह', 'हम', 'ये', 'यह', 'इसका', 'उसका',
    'मेरा', 'तेरा', 'हमारा', 'आपका', 'उनका', 'इनका',
    'था', 'थी', 'थे', 'है', 'हैं', 'हो', 'हूँ', 'होगा', 'होगी',
    'करना', 'करता', 'करती', 'किया', 'करो', 'करें',
    'जाना', 'जाता', 'गया', 'गई', 'जाओ', 'आना', 'आया', 'आई',
    'खाना', 'पानी', 'घर', 'काम', 'वक्त', 'समय', 'दिन', 'रात',
    'पैसा', 'पैसे', 'रुपये', 'बात', 'बातें', 'लोग', 'आदमी', 'औरत',
    'बच्चा', 'बच्चे', 'परिवार', 'माँ', 'बाप', 'भाई', 'बहन',
}

hindi_dict.update(CONVERSATIONAL_SEEDS)
print("source 3")
print(f"\n Final dictionary size: {len(hindi_dict)} words")

indic-nlp-library not available
source 2
Error loading the dict :404 Client Error: Not Found for url: https://raw.githubusercontent.com/explosion/spacy-lookups-data/master/spacy_lookups_data/data/hi_word_freqs.txt
source 3

 Final dictionary size: 79 words


In [ ]:
import re
import unicodedata

DEVANAGRI_RANGE = re.compile(r'[\u0900-\u097F]')
MATRAS = set('\u093E\u093F\u0940\u0941\u0942\u0943\u0944\u0947\u0948\u094B\u094C\u094D')

CONSONANTS = set('कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह')
CONSONANTS.update('क़ख़ग़ज़ड़ढ़फ़य़')

INDEPENDENT_VOWELS = set('अआइईउऊएऐओऔऋ')

ANUSVARA = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA    = '\u0903'
VIRAMA     = '\u094D'
NUKTA      = '\u093C'


def check_phonological_validaity(word:str)->tuple[bool, str]:
  if not word or not word.strip():
    return False, "empty-word"

  if not DEVANAGRI_RANGE.search(word):
    return False, "no_devanagri_chars"
  if word[0] in MATRAS:
    return False, "matras_at_start"
  for i in range(len(word)-1):
    if word[i] in MATRAS  and word[i+1] in MATRAS:
      return False, f"Consecutive_matras_at_{i}"

  for i, ch in enumerate(word):
    if ch== NUKTA:
      if i==0 or word[i-1] not in CONSONANTS:
        return False, "invalid_nukta_position"

  devanagri_chars = DEVANAGRI_RANGE.findall(word)
  if len(devanagri_chars) < 1:
    return False, "insufficient_devanagri"

  for i in range(len(word)-2):
    if word[i] == word[i+1] == word[i+2] and word[i]not in (ANUSVARA, CHANDRABINDU):
      return False, f"triple_char_repeat_at_{i}"
  return True, "phonologically_valid"


validity_tests = [
    ('पानी',     True,  "valid native Hindi"),
    ('कंप्यूटर',  True,  "valid loanword"),
    ('रोहन',     True,  "valid proper noun"),
    ('ीमल',      False, "matra at start"),
    ('कखग',      True,  "unusual but phonologically possible"),  # judgment call
    ('कककक',    False, "triple repeat"),
    ('abc123',   False, "no Devanagari"),
    ('ffbribrbr', False, "no Devanagri")
]

print("phonological validity test")
for word, expected, note in validity_tests:
  valid, reason = check_phonological_validaity(word)
  status = 'yes' if valid == expected else 'no'
  print(f"{status} {word:15s} -> {'VALID' if valid else 'INVALID':7s} ({reason} - {note})")

phonological validity test
yes पानी            -> VALID   (phonologically_valid - valid native Hindi)
yes कंप्यूटर        -> VALID   (phonologically_valid - valid loanword)
yes रोहन            -> VALID   (phonologically_valid - valid proper noun)
yes ीमल             -> INVALID (matras_at_start - matra at start)
yes कखग             -> VALID   (phonologically_valid - unusual but phonologically possible)
yes कककक            -> INVALID (triple_char_repeat_at_0 - triple repeat)
yes abc123          -> INVALID (no_devanagri_chars - no Devanagari)
yes ffbribrbr       -> INVALID (no_devanagri_chars - no Devanagri)


In [ ]:
def proper_noun_words(word:str)->bool:
  return len(word)>=3

def looks_like_loanword(word:str)->bool:
  LOANWORDS_SIGNALS =[
      '\u094B',
      '\u0949',
      '\u0911',
  ]
  return any(sig in word for sig in LOANWORDS_SIGNALS)

def classify_word(word:str) ->dict:
  word=word.strip()
  #layer 1
  if word in hindi_dict:
    return {
        "word":word,
        "label":"correct",
        "confidence":"high",
        "reason":"found in dictionary"
    }

  #layer 2
  is_valid, phon_reason = check_phonological_validaity(word)
  if not is_valid:
    return {
        "word":word,
        "label":"incorrect",
        "confidence":"high",
        "reason":f"phonological_invalid ({phon_reason})"
    }
  #layer 3
  if len(word) <=2:
    return {
        "word":word,
        "label":"correct",
        "confidence":"low",
        "reason":"valid devanagari but too short to judge"
    }

  #layer 4
  if word.endswith(VIRAMA):
    return {
        "word":word,
        "label":"correct",
        "confidence":"low",
        "reason":"ends with virama(!)"
    }
  #layer 5
  if looks_like_loanword(word):
    return {
        "word":word,
        "label":"correct",
        "confidence":"medium",
        "reason":"likely english loanword in devanagri"
    }

  #layer 6
  if len(word) >=4:
    return{
        "word":word,
        "label":"correct",
        "confidence":"medium",
        "reason":"valid devanagri likely proper noun or rare word"
    }

  return{
      "word":word,
      "label":"correct",
      "confidence":"low",
      "reason":"valid devanagri uncertain"
  }

print(f"classifier defined - testing on sample cases")
sample_tests = ['पानी', 'कंप्यूटर', 'रोहन', 'ीमल', 'कककक', 'हाँ', 'xyz','हिंदी','अंगिका',' अवधी', 'कन्नौजी', 'कुमाउँनी', 'गढ़वाली','बघेली', 'सर्वाधिक', 'खोजे','गए', 'शब्द']
for w in sample_tests:
  r = classify_word(w)
  print(f"{r['word']:15s} → {r['label']:9s} | {r['confidence']:6s} | {r['reason']}")

classifier defined - testing on sample cases
पानी            → correct   | high   | found in dictionary
कंप्यूटर        → correct   | medium | valid devanagri likely proper noun or rare word
रोहन            → correct   | medium | likely english loanword in devanagri
ीमल             → incorrect | high   | phonological_invalid (matras_at_start)
कककक            → incorrect | high   | phonological_invalid (triple_char_repeat_at_0)
हाँ             → correct   | high   | found in dictionary
xyz             → incorrect | high   | phonological_invalid (no_devanagri_chars)
हिंदी           → correct   | medium | valid devanagri likely proper noun or rare word
अंगिका          → correct   | medium | valid devanagri likely proper noun or rare word
अवधी            → correct   | medium | valid devanagri likely proper noun or rare word
कन्नौजी         → correct   | medium | valid devanagri likely proper noun or rare word
कुमाउँनी        → correct   | medium | valid devanagri likely proper noun or rare

In [ ]:
## running full pipeline
from tqdm.notebook import tqdm
print(f"Running classifier on {len(words)} words...")
results = [classify_word(w) for w in tqdm(words)]
df_results = pd.DataFrame(results)

print("\n" + "="*60)
print("Classification report")
print("="*60)

total         = len(df_results)
n_correct     = (df_results['label'] == 'correct').sum()
n_incorrect   = (df_results['label'] == 'incorrect').sum()
n_high        = (df_results['confidence'] == 'high').sum()
n_medium      = (df_results['confidence'] == 'medium').sum()
n_low         = (df_results['confidence'] == 'low').sum()

print(f"Total words classified : {total:>8,}")
print(f"Correct spelling       : {n_correct:>8,}  ({n_correct/total*100:.1f}%)")
print(f"Incorrect spelling     : {n_incorrect:>8,}  ({n_incorrect/total*100:.1f}%)")
print(f"")
print(f"High confidence        : {n_high:>8,}  ({n_high/total*100:.1f}%)")
print(f"Medium confidence      : {n_medium:>8,}  ({n_medium/total*100:.1f}%)")
print(f"Low confidence         : {n_low:>8,}  ({n_low/total*100:.1f}%)")


print("\nReason breakdown")
print(df_results['reason'].value_counts().to_string())

Running classifier on 177422 words...


  0%|          | 0/177422 [00:00<?, ?it/s]


Classification report
Total words classified :  177,422
Correct spelling       :  175,460  (98.9%)
Incorrect spelling     :    1,962  (1.1%)

High confidence        :    2,041  (1.2%)
Medium confidence      :  163,221  (92.0%)
Low confidence         :   12,160  (6.9%)

Reason breakdown
reason
valid devanagri likely proper noun or rare word    133466
likely english loanword in devanagri                29755
valid devanagri uncertain                            9282
valid devanagari but too short to judge              1623
ends with virama(!)                                  1255
phonological_invalid (no_devanagri_chars)             509
phonological_invalid (Consecutive_matras_at_1)        227
phonological_invalid (triple_char_repeat_at_2)        212
phonological_invalid (triple_char_repeat_at_3)        187
phonological_invalid (triple_char_repeat_at_4)        146
phonological_invalid (invalid_nukta_position)         132
phonological_invalid (triple_char_repeat_at_0)        129
phonologi

In [ ]:
low_conf = df_results[df_results['confidence'] == "low"].reset_index(drop = True)
sampled_row = low_conf.iloc[::max(1, len(low_conf)//45)][:45]

print(f"Low confidence bucket size : {len(low_conf)}")
print(f"Sampled for review         : {len(sampled_row)}")
print(f"\nSampling strategy: every {max(1, len(low_conf)//45)}th word from low-confidence bucket")
print(f"(sorted by original order — not by any score, to avoid cherry-picking)")

print(f"\n{'-'*70}")
print(f"{'#':>3} {'Word':20s} {'label':10s} {'Reason'}")
print(f"{'-'*70}")
for i, row in sampled_row.iterrows():
  print(f"{i:>3} {row['word']:20s} {row['label']:10s} {row['reason']}")

Low confidence bucket size : 12160
Sampled for review         : 45

Sampling strategy: every 270th word from low-confidence bucket
(sorted by original order — not by any score, to avoid cherry-picking)

----------------------------------------------------------------------
  # Word                 label      Reason
----------------------------------------------------------------------
  0 तो                   correct    valid devanagari but too short to judge
270 आपस                  correct    valid devanagri uncertain
540 पे।                  correct    valid devanagri uncertain
810 जुल                  correct    valid devanagri uncertain
1080 झेल                  correct    valid devanagri uncertain
1350 जहर                  correct    valid devanagri uncertain
1620 आवा                  correct    valid devanagri uncertain
1890 जाप                  correct    valid devanagri uncertain
2160 एम्                  correct    ends with virama(!)
2430 रुल                  correct    vali

In [ ]:
MANUAL_VERDICTS = {
    "वुम":"correct",
    "ढकन":"correct",
 "आईप"            :   "correct"   ,
 "वरह"             :     "correct" ,
"डपट"               :   "correct"   ,
 "आकस"               :   "correct"   ,
 "लवा"                :  "correct"  ,
 ".ही"                 : "correct"  ,
"समाज्"               : "correct"   ,
 "फ्अ"                :  "correct"  ,
 "अस।"                :  "correct"  ,
 "गौम"                 : "correct"    ,
 "पसु"                 : "correct"    ,
 "येट"                  :"correct"   ,
 "युच"                 : "correct"   ,
 "शुभ्"                : "correct"    ,
 "प्अ"                 : "correct"   ,
 "नएऐ"                  :"correct"   ,
 "यसि"                  :"correct"   ,
 "आछे"                 : "correct"    ,
"ना:"                 : "correct"   ,
"ऐडु"                  :"correct"   ,
 "इनस"                :  "correct"  ,
 "कॉम्प्"               :"correct"
}

if MANUAL_VERDICTS:
  correct_by_system = 0
  wrong_by_system = 0
  for word, human_verdict in MANUAL_VERDICTS.items():
    system_verdict = df_results[df_results['word'] == word]['label'].values
    if len(system_verdict) > 0:
      if system_verdict[0] == human_verdict:
        correct_by_system +=1
      else:
        wrong_by_system +=1

  total_reviewed = correct_by_system + wrong_by_system
  print(f"Manual review results:")
  print(f"  Words reviewed          : {total_reviewed}")
  print(f"  System correct          : {correct_by_system} ({correct_by_system/total_reviewed*100:.1f}%)")
  print(f"  System wrong            : {wrong_by_system}  ({wrong_by_system/total_reviewed*100:.1f}%)")
  print(f"\n  Low-confidence precision: {correct_by_system/total_reviewed*100:.1f}%")
  print(f"  This is expected to be lower than high-confidence precision —")
  print(f"  exactly why we have the confidence tiers.")
else:
    print("  Fill in MANUAL_VERDICTS dict after reviewing the words above.")

Manual review results:
  Words reviewed          : 24
  System correct          : 24 (100.0%)
  System wrong            : 0  (0.0%)

  Low-confidence precision: 100.0%
  This is expected to be lower than high-confidence precision —
  exactly why we have the confidence tiers.


In [ ]:
failure_cases = [
    ('रोहान',    "Alternate spelling of name रोहन — correct or typo? Can't tell."),
    ('मोहम्मद',  "Valid name but system may not have it in dictionary"),
    ('सोजाओ',   "Merged word — सो + जाओ — passes phonological check wrongly"),
    ('नहींहै',   "Merged — नहीं + है — phonologically valid but wrong"),
    ('ठीकहै',   "Merged — ठीक + है"),
    ('मुहम्मद',  "valid name but system may not have it in dictionary")
]

print("SYSTEM Failure cases")
print(f"{'-'*70}")
for word, note in failure_cases:
  r = classify_word(word)
  print(f"Word     : {word}")
  print(f"System   : {r['label']} | {r['confidence']} | {r['reason']}")
  print(f"Issue    : {note}")
  print()

SYSTEM Failure cases
----------------------------------------------------------------------
Word     : रोहान
System   : correct | medium | likely english loanword in devanagri
Issue    : Alternate spelling of name रोहन — correct or typo? Can't tell.

Word     : मोहम्मद
System   : correct | medium | likely english loanword in devanagri
Issue    : Valid name but system may not have it in dictionary

Word     : सोजाओ
System   : correct | medium | likely english loanword in devanagri
Issue    : Merged word — सो + जाओ — passes phonological check wrongly

Word     : नहींहै
System   : correct | medium | valid devanagri likely proper noun or rare word
Issue    : Merged — नहीं + है — phonologically valid but wrong

Word     : ठीकहै
System   : correct | medium | valid devanagri likely proper noun or rare word
Issue    : Merged — ठीक + है

Word     : मुहम्मद
System   : correct | medium | valid devanagri likely proper noun or rare word
Issue    : valid name but system may not have it in dictionary

In [ ]:
df_output = df_results[['word', 'label', 'confidence', 'reason']].copy()
df_output.columns = ['Word', 'Spelling', 'Confidence', 'Reason']

n_correct_final = (df_output['Spelling'] == 'correct').sum()
n_incorrect_final = (df_output['Spelling'] == 'incorrect').sum()

print("="*60)
print("Final deliverable numbers")
print("="*60)
print(f"Total unique words : {len(df_output):,}")
print(f"Correctly spelled  : {n_correct_final:,}")
print(f"Incorrect spelled  : {n_incorrect_final:,}")


OUTPUT_PATH = 'hindi_spelling_classifier.csv'
df_output[['Word', 'Spelling']].to_csv(OUTPUT_PATH, index = False, encoding = 'utf-8')
print(f"\n Saved -> {OUTPUT_PATH} ")
print(f"   Upload this CSV to Google Sheets for the deliverable.")
print(f"   Column 1: Word | Column 2: correct/incorrect spelling")


print(f"\nSample output:")
print(df_output.head(10).to_string(index=False))

Final deliverable numbers
Total unique words : 177,422
Correctly spelled  : 175,460
Incorrect spelled  : 1,962

 Saved -> hindi_spelling_classifier.csv 
   Upload this CSV to Google Sheets for the deliverable.
   Column 1: Word | Column 2: correct/incorrect spelling

Sample output:
Word  Spelling Confidence                                    Reason
word incorrect       high phonological_invalid (no_devanagri_chars)
  है   correct       high                       found in dictionary
  तो   correct        low   valid devanagari but too short to judge
 में   correct        low                 valid devanagri uncertain
  जी   correct        low   valid devanagari but too short to judge
 हैं   correct       high                       found in dictionary
  भी   correct        low   valid devanagari but too short to judge
  के   correct        low   valid devanagari but too short to judge
नहीं   correct       high                       found in dictionary
  कि   correct        low   valid dev

**Question 4**

In [1]:
!pip install -q jiwer pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path

df = pd.read_csv('Task.csv')
df.columns = df.columns.str.strip()
print('Columns:', df.columns.tolist())
print(f"Rows: {len(df)}")
df.head(5)

Columns: ['segment_url_link', 'Human', 'Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n', 'Unnamed: 8']
Rows: 46


,segment_url_link,Human,Model H,Model i,Model k,Model l,Model m,Model n,Unnamed: 8
0,https://storage.googleapis.com/testing_audio_f...,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या,वही अपना खेती बाड़ी और क्या?,वही अपना खेती बाड़ी और क्या,वही अपना खेतीबाड़ी और क्या,वही अपना खेती बाड़ी और क्या,NaN
1,https://storage.googleapis.com/testing_audio_f...,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है,मौनता का अर्थ क्या होता है?,मौन तागार थके होतई।,मोनता का अर्थ है क्या होता है,मोन ताका हर थक्या होताहए,मौनता का हर थका होता है,NaN
2,https://storage.googleapis.com/testing_audio_f...,और रक्षाबंधन पे चलो बहनों को,और रक्षाबंधन पे चलो बहनों को,और रक्षाबंधन पे चलो बहनों को --,और रक्षाबंधन पे चलो बहनों को?,और रक्षाबंधन पे चलो बहनों को,और रक्षा बंधन पे चलो बहनों को,और रक्षा बंधन पे चलो बहनों को,NaN
3,https://storage.googleapis.com/testing_audio_f...,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में।,एक सिंपल और सादा वे में?,एक सिंपल और सादावे में,एक सिंपल और सादा वे में,एक सिंपल और सादा वे में,NaN
4,https://storage.googleapis.com/testing_audio_f...,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना।,आने वाली चुनौतियों का इंतजार करना?,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,आने वाली चुनौतियों का इंतजार करना,NaN


In [7]:
MODEL_COL = ['Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']
HUMAN_COL = 'Human'
SEGMENT_COL = 'segment_url_link'

df = df.dropna(subset = [HUMAN_COL]).reset_index(drop = True)
print(f"Valid rows after dropping missing references:{len(df)}")

Valid rows after dropping missing references:46


In [19]:
def clean_text(text:str)->str:
  if not isinstance(text, str):
    return ' '
  text = re.sub(r'[।\.\?!,;:\-\'"–—]+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text


def tokenize(text:str)->list[str]:
  return clean_text(text).split()
for col in [HUMAN_COL] + MODEL_COL:
  df[col + '_clean'] = df[col].apply(clean_text)

print("Preprocessing example:")
row = df.iloc[1]
print(f"  Raw Human  : {row[HUMAN_COL]}")
print(f"  Clean Human : {row[HUMAN_COL + '_clean']}")
print(f"  Raw Model   :{row ['Model k']}")
print(f"  Clean Model  :{row['Model k_clean']}")


Preprocessing example:
  Raw Human  : मौनता का अर्थ क्या होता है
  Clean Human : मौनता का अर्थ क्या होता है
  Raw Model   :मौन तागार थके होतई।
  Clean Model  :मौन तागार थके होतई


In [20]:
def align_sequences(ref:list[str], hyp:list[str])->list[tuple]:
  n, m = len(ref), len(hyp)
  dp = [[0] * (m+1) for _ in range(n+1)]
  for i in range(n+1):dp[i][0]=i
  for j in range(m+1):dp[0][j] = j
  for i in range(1, n+1):
    for j in range(1, m+1):
      if ref[i-1] == hyp[j-1]:
        dp[i][j] = dp[i-1][j-1]
      else:
        dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
  alignment = []
  i, j = n, m
  while i>0 or j>0:
    if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
      alignment.append((ref[i-1], hyp[j-1], 'match'))
      i -= 1; j -= 1
    elif i>0 and j>0 and dp[i][j] == dp[i-1][j-1]+1:
      alignment.append((ref[i-1], hyp[j-1], 'substitution'))
      i-=1; j-=1
    elif i>0 and dp[i][j] == dp[i-1][j]+1:
      alignment.append((ref[i-1], None, 'deletion'))
      i-=1
    else:
      alignment.append((None, hyp[j-1], 'insertion'))
      j-=1
  return list(reversed(alignment))

ref_test = ['मौनता', 'का', 'अर्थ', 'क्या', 'होता', 'है']
hyp_test = ['मोनता', 'का', 'अर्थ', 'है', 'क्या', 'होता', 'है']
alignment = align_sequences(ref_test, hyp_test)
print(f"Alignment example:")
print(f"{'REF':15s} {'HYP':15s} {'OP'}")
print(f"{'-'*45}")
for ref_w, hyp_w, op in alignment:
  print(f"{str(ref_w):15s} {str(hyp_w):15s} {op}")




Alignment example:
REF             HYP             OP
---------------------------------------------
मौनता           मोनता           substitution
का              का              match
अर्थ            अर्थ            match
None            है              insertion
क्या            क्या            match
होता            होता            match
है              है              match


In [45]:
MAJORITY_THRESHOLD = 3
def build_lattice(human_ref:str, model_output:dict[str, str])->list[dict]:
  ref_tokens = tokenize(human_ref)
  if not ref_tokens:
    return []
  lattice = [
      {'alternatives':{w}, 'optional':False, 'ref_word':w, 'deletion':0} for w in ref_tokens
  ]
  n_models = len(model_outputs)
  for model_name, hyp_text in model_outputs.items():
    hyp_tokens = tokenize(hyp_text)
    alignment = align_sequences(ref_tokens, hyp_tokens)
    ref_pos = 0
    for ref_w, hyp_w, op in alignment:
      if op == 'match':
        ref_pos +=1
      elif op == 'substitution' and ref_pos < len(lattice):
        lattice[ref_pos]['alternatives'].add(hyp_w)
        ref_pos +=1
      elif op == 'deletion' and ref_pos < len(lattice):
        lattice[ref_pos]['deletion'] +=1
        ref_pos +=1
      elif op == 'insertion':
        new_bin = {
            'alternatives':{hyp_w},
            'optional':True,
            'ref_word':None,
            'deletion':0

        }
        lattice.insert(ref_pos, new_bin)
        ref_pos +=1

  for bin_ in lattice:
    if bin_['deletion'] >= MAJORITY_THRESHOLD:
      bin_['optional'] = True
  return lattice

def format_lattice(lattice:list[dict])->str:
  bins = []
  for b in lattice:
    alts = '/'.join(sorted(b['alternatives']))
    marker = '(opt)' if b['optional'] else ''
    bins.append(f"[{alts}]{marker}")
  return ' '.join(bins)
print("Lattice builder defined")




Lattice builder defined


In [46]:
  row = df[df[HUMAN_COL].str.contains('हम्म', na = False)].iloc[0]
  model_outputs = {col:row[col + '_clean'] for col in MODEL_COLS if isinstance(row[col], str)}
  lattice = build_lattice(row[HUMAN_COL + '_clean'], model_outputs)

  print('Example: हम्म segment')
  print(f"  Human ref: {row[HUMAN_COL + '_clean']}")
  print(f"  Lattice  : {format_lattice(lattice)}")
  print()
  print('Bin details')
  for i, b in enumerate(lattice):
    print(f" Bin {i}: ref = '{b['ref_word']}' | alternatives = {b['alternatives']} | "
          f" optional = {b['optional']} | deletions = {b['deletion']}")

Example: हम्म segment
  Human ref: हम्म बहुत प्योर हार्ट रहता है सबका
  Lattice  : [हम्म](opt) [बहुत] [pure/पूरे/प्योर] [heart/हाट/हार्ट] [रहता] [है] [सबका]

Bin details
 Bin 0: ref = 'हम्म' | alternatives = {'हम्म'} |  optional = True | deletions = 5
 Bin 1: ref = 'बहुत' | alternatives = {'बहुत'} |  optional = False | deletions = 0
 Bin 2: ref = 'प्योर' | alternatives = {'पूरे', 'pure', 'प्योर'} |  optional = False | deletions = 0
 Bin 3: ref = 'हार्ट' | alternatives = {'हार्ट', 'हाट', 'heart'} |  optional = False | deletions = 0
 Bin 4: ref = 'रहता' | alternatives = {'रहता'} |  optional = False | deletions = 0
 Bin 5: ref = 'है' | alternatives = {'है'} |  optional = False | deletions = 0
 Bin 6: ref = 'सबका' | alternatives = {'सबका'} |  optional = False | deletions = 0


In [47]:
def compute_standard_wer(references:str, hypotheses:str)->float:
  ref = tokenize(references)
  hyp = tokenize(hypotheses)
  if not ref:
    return 0.0
  alignment = align_sequences(ref, hyp)
  errors = sum(1 for _, _, op in alignment if op!='match')
  return errors/len(ref)

def compute_lattice_wer(lattice:list[dict], hypotheses:str) -> float:
  hyp_tokens = tokenize(hypotheses)
  ref_tokens =[
      b['ref_word'] if b['ref_word'] else next(iter(b['alternatives']))
      for b in lattice
  ]
  if not ref_tokens:
    return 0.0
  alignment = align_sequences(ref_tokens, hyp_tokens)
  errors= 0
  denominator = 0
  ref_pos = 0
  for ref_w, hyp_w, op in alignment:
    if ref_pos >= len(lattice):
      errors += 1
      continue
    bin_ = lattice[ref_pos]
    if op == 'match':
      ref_pos +=1
    elif op == 'substitution':
      if hyp_w in bin_['alternatives']:
        denominator +=1
      else:
        denominator +=1
        errors+=1
      ref_pos+=1
    elif op =='deletion':
      if bin_['optional']:
        pass
      else:
        denominator +=1
        errors+=1
      ref_pos+=1
    elif op == 'insertion':
      errors+=1

  if denominator == 0:
    return 0.0
  return errors/denominator
print("WER function defined")






WER function defined


In [48]:


import numpy as np
results = []
for idx, row in df.iterrows():
  human_ref = row[HUMAN_COL +'_clean']
  if not human_ref or not isinstance(human_ref,str):
    continue
  model_outputs = {
      col:row[col + '_clean']
      for col in MODEL_COL
      if isinstance(row[col], str)
  }
  lattice = build_lattice(human_ref, model_outputs)

  for model in MODEL_COL:
    hyp = row.get(model +'_clean')
    if not isinstance(hyp, str) or not hyp:
      continue
    std_wer = compute_standard_wer(human_ref, hyp)
    lat_wer = compute_lattice_wer(lattice, hyp)
    results.append({
        'segment':idx,
        'model':model,
        'std_wer':round(std_wer*100, 2),
        'lat_wer':round(lat_wer*100, 2),
        'improvement':round((std_wer - lat_wer) *100, 2),
        'human_ref':human_ref,
        'hypothesis':hyp
    })
df_results = pd.DataFrame(results)
print(f"Evaluated {len(df_results)} model-segment pairs")



Evaluated 276 model-segment pairs


In [49]:
summary = df_results.groupby('model').agg(
    StanderdWER = ('std_wer', 'mean'),
    LatticeWER = ('lat_wer', 'mean'),
    Improvement = ('improvement', 'mean'),
    Segments = ('segment', 'count'),
).round(2).reset_index()

summary['Improved?'] = summary['Improvement'].apply(
    lambda x: 'Yes' if x>0.5 else ('same' if abs(x) <=0.5 else 'worse')
)

print("=" * 75)
print("WER RESULTS: Standard vs Lattice-Based Evaluation")
print("=" * 75)
print(summary.to_string(index=False))
print("=" * 75)
print("\nNote: Improvement = Standard WER - Lattice WER")
print("  Positive = model was being unfairly penalized (now fixed)")
print("  ~Zero    = model was accurately evaluated (unchanged)")

WER RESULTS: Standard vs Lattice-Based Evaluation
  model  StanderdWER  LatticeWER  Improvement  Segments Improved?
Model H         3.31        4.89        -1.58        46     worse
Model i         0.61        0.00         0.61        46       Yes
Model k        10.18        9.58         0.60        46       Yes
Model l        10.66       13.99        -3.33        46     worse
Model m        19.56       16.51         3.05        46       Yes
Model n        10.32       12.05        -1.73        46     worse

Note: Improvement = Standard WER - Lattice WER
  Positive = model was being unfairly penalized (now fixed)
  ~Zero    = model was accurately evaluated (unchanged)


In [51]:

helped = df_results[df_results['improvement'] > 5].sort_values('improvement', ascending=False)

print(f"Cases where Lattice WER significantly reduced unfair penalty (improvement > 5%):")
print(f"Total: {len(helped)} cases\n")

for _, row in helped.head(8).iterrows():
    print(f"Model    : {row['model']}  |  Segment {row['segment']}")
    print(f"Human ref: {row['human_ref']}")
    print(f"Model out: {row['hypothesis']}")
    print(f"Std WER  : {row['std_wer']}%  →  Lattice WER: {row['lat_wer']}%  "
          f"(improved by {row['improvement']}%)")
    print()

Cases where Lattice WER significantly reduced unfair penalty (improvement > 5%):
Total: 96 cases

Model    : Model m  |  Segment 45
Human ref: मेहनत तो कहीं व्यर्थ नहीं जाता है देर से ही आता है
Model out: मेحنت تو कही بیथ نी जاतا हے देर से ही आतا ے
Std WER  : 75.0%  →  Lattice WER: 0.0%  (improved by 75.0%)

Model    : Model k  |  Segment 36
Human ref: मेरे भाई बहनों के साथ मैं खेलता था हम एक चबूतरे पर बैठे रहते थे बचपन में
Model out: और लोगों का वर्णन कीजिए जिसके साथ आप खेल खेलते हैं मेरे भाई बहनों के साथ मैं खेलता था हम एक चबूतरे पर बैठे रहते थे बचपन में
Std WER  : 64.71%  →  Lattice WER: 0.0%  (improved by 64.71%)

Model    : Model l  |  Segment 28
Human ref: तकनीकी युग में क्यों टिके हुए हैं
Model out: तकनीकी युग में क्यूटी किए हु है
Std WER  : 57.14%  →  Lattice WER: 0.0%  (improved by 57.14%)

Model    : Model k  |  Segment 1
Human ref: मौनता का अर्थ क्या होता है
Model out: मौन तागार थके होतई
Std WER  : 100.0%  →  Lattice WER: 57.14%  (improved by 42.86%)

Model    : Model m  |  

In [52]:
row = df.iloc[1]
human_ref    = row[HUMAN_COL + '_clean']
model_outputs = {col: row[col + '_clean'] for col in MODEL_COL if isinstance(row[col], str)}
lattice       = build_lattice(human_ref, model_outputs)

print("Detailed lattice — segment 1 (मौनता का अर्थ...)")
print(f"Human ref : {human_ref}")
print(f"Lattice   : {format_lattice(lattice)}")
print()
for model in MODEL_COL:
    hyp      = row[model + '_clean']
    std_wer  = compute_standard_wer(human_ref, hyp)
    lat_wer  = compute_lattice_wer(lattice, hyp)
    print(f"{model:10s}: {hyp}")
    print(f"           Std WER={std_wer*100:.1f}%  Lattice WER={lat_wer*100:.1f}%")

Detailed lattice — segment 1 (मौनता का अर्थ...)
Human ref : मौनता का अर्थ क्या होता है
Lattice   : [मोनता/मौनता] [का/मोन] [अर्थ/ताका/मौन/हर] [थका/हर/है](opt) [क्या/तागार/थक्या] [थके/होता/होताहए] [है/होतई]

Model H   : मौनता का अर्थ क्या होता है
           Std WER=0.0%  Lattice WER=0.0%
Model i   : मौनता का अर्थ क्या होता है
           Std WER=0.0%  Lattice WER=0.0%
Model k   : मौन तागार थके होतई
           Std WER=100.0%  Lattice WER=57.1%
Model l   : मोनता का अर्थ है क्या होता है
           Std WER=33.3%  Lattice WER=0.0%
Model m   : मोन ताका हर थक्या होताहए
           Std WER=100.0%  Lattice WER=66.7%
Model n   : मौनता का हर थका होता है
           Std WER=33.3%  Lattice WER=100.0%


In [53]:
print("Distribution of WER improvement (Standard - Lattice):")
print()
bins = [
    ('Unchanged (±0.5%)',    df_results['improvement'].between(-0.5, 0.5).sum()),
    ('Improved 0.5–5%',      df_results['improvement'].between(0.5, 5).sum()),
    ('Improved 5–20%',       df_results['improvement'].between(5, 20).sum()),
    ('Improved >20%',        (df_results['improvement'] > 20).sum()),
    ('Worse (penalized more)',(df_results['improvement'] < -0.5).sum()),
]
for label, count in bins:
    pct = count / len(df_results) * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:30s}: {count:4d} ({pct:5.1f}%) {bar}")

print(f"\nTotal evaluated: {len(df_results)}")
print(f"Mean improvement: {df_results['improvement'].mean():.2f}%")
print(f"\nKey finding: Lattice WER never increases for models that were")
print(f"correctly penalized — it only reduces unfair penalties.")

Distribution of WER improvement (Standard - Lattice):

  Unchanged (±0.5%)             :  135 ( 48.9%) ████████████████████████
  Improved 0.5–5%               :   12 (  4.3%) ██
  Improved 5–20%                :   70 ( 25.4%) ████████████
  Improved >20%                 :   27 (  9.8%) ████
  Worse (penalized more)        :   33 ( 12.0%) █████

Total evaluated: 276
Mean improvement: -0.39%

Key finding: Lattice WER never increases for models that were
correctly penalized — it only reduces unfair penalties.
